In [1]:
import pandas as pd
import numpy as np

TRAIN_PATH = "../data/raw/fraudTrain.csv"
TEST_PATH = "../data/raw/fraudTest.csv"

In [2]:
USE_COLS = [
    "trans_date_trans_time",
    "cc_num",
    "category",
    "amt",
    "gender",
    "city_pop",
    "dob",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "is_fraud"
]

train = pd.read_csv(
    TRAIN_PATH,
    usecols=USE_COLS
)

print(train.shape)
train.head()

(1296675, 12)


,trans_date_trans_time,cc_num,category,amt,gender,lat,long,city_pop,dob,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,misc_net,4.97,F,36.0788,-81.1781,3495,1988-03-09,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,grocery_pos,107.23,F,48.8878,-118.2105,149,1978-06-21,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,entertainment,220.11,M,42.1808,-112.2620,4154,1962-01-19,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,gas_transport,45.00,M,46.2306,-112.1138,1939,1967-01-12,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,misc_pos,41.96,M,38.4207,-79.4629,99,1986-03-28,38.674999,-78.632459,0


# Convert dates

In [3]:
train["trans_date_trans_time"] = pd.to_datetime(
    train["trans_date_trans_time"]
)

train["dob"] = pd.to_datetime(
    train["dob"]
)

train = train.sort_values(
    "trans_date_trans_time"
).reset_index(drop=True)

# Temporal features

In [4]:
train["transaction_hour"] = train["trans_date_trans_time"].dt.hour

train["day_of_week"] = train["trans_date_trans_time"].dt.dayofweek

train["is_weekend"] = (
    train["day_of_week"] >= 5
).astype(int)

train["is_night"] = (
    (train["transaction_hour"] >= 23) |
    (train["transaction_hour"] <= 5)
).astype(int)

# Age

In [5]:
train["age"] = (
    (
        train["trans_date_trans_time"] -
        train["dob"]
    ).dt.days / 365.25
).astype(int)

In [6]:
train["age"].describe()

count    1.296675e+06
mean     4.549541e+01
std      1.739698e+01
min      1.300000e+01
25%      3.200000e+01
50%      4.300000e+01
75%      5.700000e+01
max      9.500000e+01
Name: age, dtype: float64

# Geographic distance 🌍

In [7]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [8]:
train["distance_km"] = haversine_distance(
    train["lat"],
    train["long"],
    train["merch_lat"],
    train["merch_long"]
)

In [9]:
train["distance_km"].describe()

count    1.296675e+06
mean     7.611465e+01
std      2.911693e+01
min      2.225452e-02
25%      5.533491e+01
50%      7.823175e+01
75%      9.850327e+01
max      1.521172e+02
Name: distance_km, dtype: float64

# Behavioral features

In [10]:
customer = train.groupby("cc_num", sort=False)

In [11]:
train["customer_txn_count"] = customer.cumcount()

In [12]:
train["customer_avg_amount"] = (
    customer["amt"]
    .transform(lambda x: x.shift().expanding().mean())
)

In [13]:
global_median_amt = train["amt"].median()

train["customer_avg_amount"] = (
    train["customer_avg_amount"]
    .fillna(global_median_amt)
)

# Amount ratio

In [14]:
train["amount_ratio"] = (
    train["amt"] /
    (train["customer_avg_amount"] + 1e-6)
)

# Time since previous transaction

In [15]:
train["time_since_last_txn"] = (
    customer["trans_date_trans_time"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

In [16]:
train["time_since_last_txn"] = (
    train["time_since_last_txn"]
    .fillna(-1)
)

# Inspect engineered features

In [17]:
ENGINEERED_FEATURES = [
    "amt",
    "category",
    "gender",
    "city_pop",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "age",
    "distance_km",
    "customer_txn_count",
    "customer_avg_amount",
    "amount_ratio",
    "time_since_last_txn",
    "is_fraud"
]

train[ENGINEERED_FEATURES].head(20)

,amt,category,gender,city_pop,transaction_hour,day_of_week,is_weekend,is_night,age,distance_km,customer_txn_count,customer_avg_amount,amount_ratio,time_since_last_txn,is_fraud
0,4.97,misc_net,F,3495,0,1,0,1,30,78.597568,0,47.52,0.104588,-1.0,0
1,107.23,grocery_pos,F,149,0,1,0,1,40,30.212176,0,47.52,2.256524,-1.0,0
2,220.11,entertainment,M,4154,0,1,0,1,56,108.206083,0,47.52,4.631944,-1.0,0
3,45.00,gas_transport,M,1939,0,1,0,1,51,95.673231,0,47.52,0.946970,-1.0,0
4,41.96,misc_pos,M,99,0,1,0,1,32,77.556744,0,47.52,0.882997,-1.0,0
5,94.63,gas_transport,F,2158,0,1,0,1,57,85.922643,0,47.52,1.991372,-1.0,0
6,44.54,grocery_net,F,2691,0,1,0,1,25,118.119776,0,47.52,0.937290,-1.0,0
7,71.65,gas_transport,M,6018,0,1,0,1,71,12.766923,0,47.52,1.507786,-1.0,0
8,4.27,misc_pos,F,1472,0,1,0,1,77,25.270494,0,47.52,0.089857,-1.0,0
9,198.39,grocery_pos,F,151785,0,1,0,1,44,74.077750,0,47.52,4.174874,-1.0,0


In [18]:
train[[
    "age",
    "distance_km",
    "customer_avg_amount",
    "amount_ratio",
    "time_since_last_txn",
    "customer_txn_count"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
age,1296675.0,45.495406,17.396980,13.000000,32.000000,43.000000,57.000000,95.000000
distance_km,1296675.0,76.114651,29.116935,0.022255,55.334913,78.231751,98.503268,152.117173
customer_avg_amount,1296675.0,70.372301,21.075159,1.030000,58.003499,65.032608,83.829988,1433.540000
amount_ratio,1296675.0,1.021826,2.657841,0.002176,0.157215,0.667699,1.204272,676.387819
time_since_last_txn,1296675.0,541.005719,789.857881,-1.000000,99.933333,275.600000,669.916667,22357.850000
customer_txn_count,1296675.0,908.862664,677.804183,0.000000,356.000000,777.000000,1342.000000,3122.000000


In [19]:
train[ENGINEERED_FEATURES].isnull().sum()

amt                    0
category               0
gender                 0
city_pop               0
transaction_hour       0
day_of_week            0
is_weekend             0
is_night               0
age                    0
distance_km            0
customer_txn_count     0
customer_avg_amount    0
amount_ratio           0
time_since_last_txn    0
is_fraud               0
dtype: int64

In [20]:
train["has_customer_history"] = (
    train["customer_txn_count"] > 0
).astype(int)

In [21]:
train["has_customer_history"].value_counts()

has_customer_history
1    1295692
0        983
Name: count, dtype: int64

# TEST DATA FEATURE ENGINEERING

In [22]:
TEST_PATH = "../data/raw/fraudTest.csv"

test = pd.read_csv(
    TEST_PATH,
    usecols=USE_COLS
)

print(test.shape)
test.head()

(555719, 12)


,trans_date_trans_time,cc_num,category,amt,gender,lat,long,city_pop,dob,merch_lat,merch_long,is_fraud
0,2020-06-21 12:14:25,2291163933867244,personal_care,2.86,M,33.9659,-80.9355,333497,1968-03-19,33.986391,-81.200714,0
1,2020-06-21 12:14:33,3573030041201292,personal_care,29.84,F,40.3207,-110.4360,302,1990-01-17,39.450498,-109.960431,0
2,2020-06-21 12:14:53,3598215285024754,health_fitness,41.28,F,40.6729,-73.5365,34496,1970-10-21,40.495810,-74.196111,0
3,2020-06-21 12:15:15,3591919803438423,misc_pos,60.05,M,28.5697,-80.8191,54767,1987-07-25,28.812398,-80.883061,0
4,2020-06-21 12:15:17,3526826139003047,travel,3.19,M,44.2529,-85.0170,1126,1955-07-06,44.959148,-85.884734,0


In [23]:
test["trans_date_trans_time"] = pd.to_datetime(
    test["trans_date_trans_time"]
)

test["dob"] = pd.to_datetime(test["dob"])

test = test.sort_values(
    "trans_date_trans_time"
).reset_index(drop=True)

In [24]:
print("Start:", test["trans_date_trans_time"].min())
print("End:  ", test["trans_date_trans_time"].max())

Start: 2020-06-21 12:14:25
End:   2020-12-31 23:59:34


In [25]:
test["transaction_hour"] = (
    test["trans_date_trans_time"].dt.hour
)

test["day_of_week"] = (
    test["trans_date_trans_time"].dt.dayofweek
)

test["is_weekend"] = (
    test["day_of_week"] >= 5
).astype(int)

test["is_night"] = (
    (test["transaction_hour"] >= 23) |
    (test["transaction_hour"] <= 5)
).astype(int)

In [26]:
test["age"] = (
    (
        test["trans_date_trans_time"] -
        test["dob"]
    ).dt.days / 365.25
).astype(int)

In [27]:
test["distance_km"] = haversine_distance(
    test["lat"],
    test["long"],
    test["merch_lat"],
    test["merch_long"]
)

In [28]:
test.groupby("cc_num")

In [29]:
history = (
    train.groupby("cc_num")
    .agg(
        customer_txn_count=("amt", "count"),
        customer_avg_amount=("amt", "mean"),
        last_transaction_time=("trans_date_trans_time", "max")
    )
    .to_dict("index")
)

In [30]:
test_customer_count = []
test_customer_avg = []
test_time_since = []
test_has_history = []

for _, row in test.iterrows():

    card = row["cc_num"]

    if card in history:
        state = history[card]

        previous_count = state["customer_txn_count"]
        previous_avg = state["customer_avg_amount"]
        previous_time = state["last_transaction_time"]

        time_since = (
            row["trans_date_trans_time"] - previous_time
        ).total_seconds() / 60

        test_customer_count.append(previous_count)
        test_customer_avg.append(previous_avg)
        test_time_since.append(time_since)
        test_has_history.append(1)

        # Update history AFTER processing current transaction
        new_count = previous_count + 1

        new_avg = (
            previous_avg * previous_count +
            row["amt"]
        ) / new_count

        history[card] = {
            "customer_txn_count": new_count,
            "customer_avg_amount": new_avg,
            "last_transaction_time":
                row["trans_date_trans_time"]
        }

    else:
        # First-ever transaction for this card
        test_customer_count.append(0)
        test_customer_avg.append(train["amt"].median())
        test_time_since.append(-1)
        test_has_history.append(0)

        history[card] = {
            "customer_txn_count": 1,
            "customer_avg_amount": row["amt"],
            "last_transaction_time":
                row["trans_date_trans_time"]
        }

In [31]:
test["customer_txn_count"] = test_customer_count

test["customer_avg_amount"] = test_customer_avg

test["time_since_last_txn"] = test_time_since

test["has_customer_history"] = test_has_history

In [32]:
test["amount_ratio"] = (
    test["amt"] /
    (test["customer_avg_amount"] + 1e-6)
)

In [33]:
test[
    [
        "cc_num",
        "amt",
        "customer_txn_count",
        "customer_avg_amount",
        "amount_ratio",
        "time_since_last_txn",
        "has_customer_history"
    ]
].head(20)

,cc_num,amt,customer_txn_count,customer_avg_amount,amount_ratio,time_since_last_txn,has_customer_history
0,2291163933867244,2.86,1561,70.302293,0.040681,1522.383333,1
1,3573030041201292,29.84,2089,61.099387,0.488385,392.800000,1
2,3598215285024754,41.28,2577,89.537594,0.461035,249.433333,1
3,3591919803438423,60.05,1526,57.897438,1.037179,1197.750000,1
4,3526826139003047,3.19,2034,63.735964,0.050050,168.700000,1
5,30407675418785,19.55,2048,65.861294,0.296836,160.933333,1
6,213180742685905,133.93,2036,59.099882,2.266164,76.583333,1
7,3589289942931264,10.37,2553,61.805362,0.167785,317.183333,1
8,3596357274378601,4.37,2070,55.754700,0.078379,424.750000,1
9,3546897637165774,66.54,2019,65.403130,1.017382,9.800000,1


In [34]:
test[
    [
        "age",
        "distance_km",
        "customer_avg_amount",
        "amount_ratio",
        "time_since_last_txn",
        "customer_txn_count"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
age,555719.0,46.390496,17.432211,15.000000,33.000000,44.000000,58.000000,96.000000
distance_km,555719.0,76.104902,29.117079,0.123883,55.286255,78.179517,98.520760,150.922504
customer_avg_amount,555719.0,70.051830,16.539337,7.840000,59.843833,64.648817,83.036293,1041.540000
amount_ratio,555719.0,0.994991,2.435350,0.009335,0.153694,0.668518,1.185592,411.545582
time_since_last_txn,555719.0,455.584681,664.194001,-1.000000,84.300000,231.233333,559.275000,18949.066667
customer_txn_count,555719.0,2208.988656,936.798257,0.000000,1469.000000,2139.000000,2845.000000,4391.000000


In [35]:
test["has_customer_history"].value_counts()

has_customer_history
1    555703
0        16
Name: count, dtype: int64

In [37]:
FEATURES = [
    "amt",
    "category",
    "gender",
    "city_pop",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "age",
    "distance_km",
    "customer_txn_count",
    "customer_avg_amount",
    "amount_ratio",
    "time_since_last_txn",
    "has_customer_history"
]

TARGET = "is_fraud"

In [38]:
test[
    [
        "age",
        "distance_km",
        "customer_avg_amount",
        "amount_ratio",
        "time_since_last_txn",
        "customer_txn_count"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
age,555719.0,46.390496,17.432211,15.000000,33.000000,44.000000,58.000000,96.000000
distance_km,555719.0,76.104902,29.117079,0.123883,55.286255,78.179517,98.520760,150.922504
customer_avg_amount,555719.0,70.051830,16.539337,7.840000,59.843833,64.648817,83.036293,1041.540000
amount_ratio,555719.0,0.994991,2.435350,0.009335,0.153694,0.668518,1.185592,411.545582
time_since_last_txn,555719.0,455.584681,664.194001,-1.000000,84.300000,231.233333,559.275000,18949.066667
customer_txn_count,555719.0,2208.988656,936.798257,0.000000,1469.000000,2139.000000,2845.000000,4391.000000


In [39]:
test["has_customer_history"].value_counts()

has_customer_history
1    555703
0        16
Name: count, dtype: int64

In [40]:
FEATURES = [
    "amt",
    "category",
    "gender",
    "city_pop",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "age",
    "distance_km",
    "customer_txn_count",
    "customer_avg_amount",
    "amount_ratio",
    "time_since_last_txn",
    "has_customer_history"
]

TARGET = "is_fraud"

In [41]:
X_train_raw = train[FEATURES].copy()
y_train = train[TARGET].copy()

X_test_raw = test[FEATURES].copy()
y_test = test[TARGET].copy()

In [42]:
print("X_train:", X_train_raw.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test_raw.shape)
print("y_test:", y_test.shape)

X_train: (1296675, 15)
y_train: (1296675,)
X_test: (555719, 15)
y_test: (555719,)


In [43]:
CATEGORICAL_FEATURES = [
    "category",
    "gender"
]

In [44]:
NUMERICAL_FEATURES = [
    "amt",
    "city_pop",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "age",
    "distance_km",
    "customer_txn_count",
    "customer_avg_amount",
    "amount_ratio",
    "time_since_last_txn",
    "has_customer_history"
]

In [46]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

In [47]:
X_train_cat = encoder.fit_transform(
    X_train_raw[CATEGORICAL_FEATURES]
)

X_test_cat = encoder.transform(
    X_test_raw[CATEGORICAL_FEATURES]
)

In [48]:
X_train_num = X_train_raw[
    NUMERICAL_FEATURES
].to_numpy()

X_test_num = X_test_raw[
    NUMERICAL_FEATURES
].to_numpy()

In [49]:
X_train_num = X_train_num.astype("float32")
X_test_num = X_test_num.astype("float32")

In [50]:
from scipy.sparse import hstack

X_train = hstack([
    X_train_num,
    X_train_cat
]).tocsr()

X_test = hstack([
    X_test_num,
    X_test_cat
]).tocsr()

In [51]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (1296675, 29)
X_test: (555719, 29)


In [52]:
print("Training target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())

Training target:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Test target:
is_fraud
0    553574
1      2145
Name: count, dtype: int64


In [53]:
print("\nTraining fraud rate:")
print(y_train.mean())

print("\nTest fraud rate:")
print(y_test.mean())


Training fraud rate:
0.005788651743883394

Test fraud rate:
0.0038598644278853163


In [54]:
import numpy as np

print(
    "Train NaN:",
    np.isnan(X_train.data).sum()
)

print(
    "Train Inf:",
    np.isinf(X_train.data).sum()
)

print(
    "Test NaN:",
    np.isnan(X_test.data).sum()
)

print(
    "Test Inf:",
    np.isinf(X_test.data).sum()
)

Train NaN: 0
Train Inf: 0
Test NaN: 0
Test Inf: 0


In [55]:
categorical_names = encoder.get_feature_names_out(
    CATEGORICAL_FEATURES
)

FEATURE_NAMES = (
    NUMERICAL_FEATURES +
    list(categorical_names)
)

print("Number of features:", len(FEATURE_NAMES))

print(FEATURE_NAMES)

Number of features: 29
['amt', 'city_pop', 'transaction_hour', 'day_of_week', 'is_weekend', 'is_night', 'age', 'distance_km', 'customer_txn_count', 'customer_avg_amount', 'amount_ratio', 'time_since_last_txn', 'has_customer_history', 'category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel', 'gender_F', 'gender_M']


In [56]:
import joblib

joblib.dump(
    encoder,
    "../models/feature_encoder.joblib"
)

['../models/feature_encoder.joblib']

In [57]:
from scipy.sparse import save_npz

save_npz(
    "../data/processed/X_train.npz",
    X_train
)

save_npz(
    "../data/processed/X_test.npz",
    X_test
)

In [58]:
y_train.to_numpy().astype("int8").tofile(
    "../data/processed/y_train.bin"
)

y_test.to_numpy().astype("int8").tofile(
    "../data/processed/y_test.bin"
)

In [59]:
import json

with open(
    "../data/processed/feature_names.json",
    "w"
) as f:
    json.dump(FEATURE_NAMES, f, indent=2)

In [60]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (1296675, 29)
X_test: (555719, 29)


In [61]:
print("Number of features:", len(FEATURE_NAMES))

Number of features: 29


In [62]:
print("Train NaN:", np.isnan(X_train.data).sum())
print("Train Inf:", np.isinf(X_train.data).sum())
print("Test NaN:", np.isnan(X_test.data).sum())
print("Test Inf:", np.isinf(X_test.data).sum())

Train NaN: 0
Train Inf: 0
Test NaN: 0
Test Inf: 0
